# DWCLP: 29 layers to 15, then 4 × 128 MLP

The saved Qwen hidden states have shape 29 × 3,584. A three-layer window with stride 2 retains central layers 0, 2, ..., 28. Each central activation is averaged with distance-weighted contributions from its adjacent layers. A fixed Gaussian kernel over hidden positions uses radius 2 and sigma 1; center contribution is 1 and each adjacent-layer contribution is 0.5, normalized by 2. Edge layers replicate the nearest available layer. Flatten the resulting 15 × 3,584 tensor into the four-hidden-layer, width-128 MLP. Training and evaluation match the raw-concat 4×128 experiment.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout 84a45a0
!pip install -q -r requirements-kaggle.txt

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
OUTPUT_DIR = Path('/kaggle/working/sala_g14_6src_dwclp_4x128')
command = [sys.executable, '/kaggle/working/SALA/sala_raw_concat.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(OUTPUT_DIR), '--bleurt-threshold', '0.5', '--epochs-erm', '40', '--seed', '42', '--train-device', 'cuda:0', '--hidden-dim', '128', '--hidden-layers', '4', '--pooling', 'dwclp', '--pool-kernel-size', '3', '--pool-stride', '2', '--dwclp-radius', '2', '--dwclp-sigma', '1.0']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / 'results.csv')
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
print('Pooled layers:', summary['pooled_layers'], 'MLP input:', summary['mlp_input_dimension'])
print(json.dumps(summary['target_metrics'], indent=2))
display(results.head())